# RAG sobre datos personales y el desarrollo de la IA

## Documentos

Las fuentes, fechas y sumas de verificación están en [`data/manifest.json`](../data/manifest.json).

| Documento | Tipo | Estatus | Páginas | Caracteres | Tablas |
|---|---|---|---:|---:|---:|
| LFPDPPP | Ley | Vigente; publicada en 2025, última reforma el 14-11-2025 | 24 | 66,650 | 0 |
| LGPDPPSO | Ley | Vigente; publicada en 2025, última reforma el 14-11-2025 | 37 | 136,701 | 0 |
| Reglamento de la LFPDPPP | Reglamento | Publicado como texto vigente; redactado para la ley de 2010 | 36 | 102,994 | 0 |
| Recomendaciones del INAI sobre datos personales e IA | Guía | No vinculante; edición de junio de 2024 | 48 | 93,635 | 26 |
| Evaluación de preparación en IA de UNESCO para México | Informe | No vinculante; 2024 | 74 | 201,353 | 2 |
| Principios de Chapultepec | Declaración | No vinculante; enero de 2026 | 2 | 7,425 | 0 |
| Iniciativa de Ley General para el Uso y Control de la IA | Iniciativa | Propuesta; no es ley vigente | 42 | 82,119 | 0 |
| **Total** | | | **263** | **690,877** | **28** |

Con fragmentos de 1,000 caracteres y traslape de 150, el corpus daría unos 800 chunks. Es solo una estimación: la estrategia de chunking aún no está definida.

## Por qué estos documentos

- **Tema común con traslape real.** Los derechos ARCO (_Acceso, Rectificación, Cancelación, Oposición_) aparecen en las dos leyes y el reglamento (17, 46 y 22 menciones), y el consentimiento aparece en seis de los siete documentos. La IA aparece en cuatro. Así, el retriever tiene pasajes plausibles pero incorrectos entre los cuales elegir.

- **Contraste entre estrategias de retrieval.** Hay términos exactos que deberían favorecer a BM25 (`ARCO`, `LFPDPPP`, números de artículo) y conceptos que deberían favorecer a los embeddings, como pedir a una empresa que borre tus datos.
- **Confusión difícil.** La LFPDPPP y la LGPDPPSO comparten mucho vocabulario, pero una regula a particulares y la otra a sujetos obligados del sector público.
- **Autoridad distinta.** La mezcla de ley, guía y propuesta permite evaluar si el sistema presenta como vigente algo que no lo es.

## Chunking

La entrada es `data/elements.jsonl`, que genera `parser.ipynb`: un registro por elemento con todas las decisiones de parsing aplicadas.

- **Flujo de lectura:** elementos con `removed` vacío, `is_note` falso y texto.
- **Notas al pie:** elementos con `removed` vacío e `is_note` verdadero.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

DATA = Path("../data")
MANIFEST = {entry["doc_id"]: entry for entry in json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))["documents"]}

elements = pd.read_json(DATA / "elements.jsonl", lines=True)
kept = elements["removed"].isna()
flow = elements[kept & ~elements["is_note"] & (elements["text"].str.strip() != "")]
notes = elements[kept & elements["is_note"]]
print(f"Flujo de lectura: {len(flow)} elementos, {flow['text'].str.len().sum():,} caracteres")
print(f"Notas al pie: {len(notes)}, {notes['text'].str.len().sum():,} caracteres")

Flujo de lectura: 3279 elementos, 682,949 caracteres
Notas al pie: 288, 46,318 caracteres


## Decisión: estrategia de chunking

Los chunks se forman juntando elementos consecutivos del flujo de lectura hasta un límite de tamaño:

- Un título de sección empieza un chunk nuevo. Los títulos seguidos («Capítulo I», «Disposiciones generales») se quedan juntos y con el texto que les sigue.
- En las leyes, el reglamento y la iniciativa, cada «Artículo N» también empieza un chunk nuevo, así que un chunk no mezcla artículos.
- Los elementos no se cortan, salvo los que son más largos que el límite (ver «Decisión: tablas largas y chunks pequeños»).
- Cada chunk guarda sus páginas del PDF, a partir de las de sus elementos, para compararlo con las respuestas esperadas.

Comparación a unos 1,000 caracteres por chunk:

| | Tamaño fijo (1,000 caracteres, traslape de 150) | Elementos agrupados |
|---|---:|---:|
| Chunks | 813 | 1,086 |
| Empiezan a mitad de un párrafo | 802 | 0 |
| Chunks legales que mezclan dos o más artículos | 174 | 0 |

Se descartaron:

- **Tamaño fijo:** corta oraciones y artículos.
- **División recursiva** (párrafo, luego oración): sigue mezclando artículos e ignora los títulos.
- **Chunking semántico:** necesita el modelo de embeddings antes de dividir y es más difícil de explicar y reproducir, cuando los documentos ya marcan su estructura.
- **El chunker de Docling:** trabaja sobre el documento original de Docling, así que ignoraría la limpieza del parsing.

In [2]:
HEADING_LABELS = {"section_header", "title"}
ARTICLE_TYPES = {"ley", "reglamento", "iniciativa de ley"}
ARTICLE_START = re.compile(r"(Art[íi]culo\s+\d+)", re.IGNORECASE)


def starts_chunk(element, doc_type):
    """A section heading, or an article in a legal text, starts a new chunk."""
    if element["label"] in HEADING_LABELS:
        return True
    return doc_type in ARTICLE_TYPES and bool(ARTICLE_START.match(element["text"]))


def current_articles(doc_flow):
    """Article in force at each element: set by an "Artículo N" element and reset by any other heading."""
    articles, current = [], None
    for label, text in zip(doc_flow["label"], doc_flow["text"]):
        match = ARTICLE_START.match(text)
        if match:
            current = match.group(1)
        elif label in HEADING_LABELS:
            current = None
        articles.append(current)
    return articles


def pack_elements(doc_flow, doc_type, max_chars):
    """Group consecutive elements into chunks of up to max_chars characters without cutting an element.
    A chunk that only holds headings stays open, so headings stay with the text that follows them."""
    groups, current = [], []
    for element in doc_flow.to_dict("records"):
        size = sum(len(item["text"]) for item in current)
        has_body = any(item["label"] not in HEADING_LABELS for item in current)
        if current and has_body and (starts_chunk(element, doc_type) or size + len(element["text"]) > max_chars):
            groups.append(current)
            current = []
        current.append(element)
    if current:
        groups.append(current)
    return groups


def chunk_record(doc_id, number, group, doc_type):
    """One chunk. `continues` marks a chunk whose first element was moved to it by the size limit,
    so it carries on an article or section that started in the previous chunk."""
    body = [item for item in group if item["label"] not in HEADING_LABELS] or group
    pages = sorted({page for item in group for page in range(int(item["page"]), int(item["page_end"]) + 1)})
    return {
        "chunk_id": f"{doc_id}-{number:04d}",
        "doc_id": doc_id,
        "pages": pages,
        "heading": body[0]["heading"],
        "n_elements": len(group),
        "continues": number > 1 and not starts_chunk(group[0], doc_type),
        "article": group[0]["article"],
        "text": "\n\n".join(item["text"] for item in group),
    }


def build_chunks(max_chars):
    records = []
    for doc_id, doc_flow in flow.groupby("doc_id", sort=False):
        doc_type = MANIFEST[doc_id]["type"]
        articles = current_articles(doc_flow) if doc_type in ARTICLE_TYPES else [None] * len(doc_flow)
        groups = pack_elements(doc_flow.assign(article=articles), doc_type, max_chars)
        records += [chunk_record(doc_id, number, group, doc_type) for number, group in enumerate(groups, start=1)]
    return pd.DataFrame(records)

## Decisión: límite de tamaño y traslape

La tabla mide los chunks con distintos límites. Los tokens se estiman con 4 caracteres por token.

In [3]:
def chunk_stats(max_chars):
    chunks = build_chunks(max_chars)
    sizes = chunks["text"].str.len()
    legal = chunks["doc_id"].map(lambda doc_id: MANIFEST[doc_id]["type"] in ARTICLE_TYPES)
    return {
        "límite": max_chars,
        "≈ tokens": max_chars // 4,
        "chunks": len(chunks),
        "mediana": int(sizes.median()),
        "p90": int(sizes.quantile(0.9)),
        "continúan un artículo o sección": int(chunks["continues"].sum()),
        "de ellos, en textos legales": int((chunks["continues"] & legal).sum()),
        "elementos más largos que el límite": int((flow["text"].str.len() > max_chars).sum()),
    }


pd.DataFrame([chunk_stats(limit) for limit in [500, 800, 1000, 1500, 2000, 3000]]).set_index("límite")

,≈ tokens,chunks,mediana,p90,continúan un artículo o sección,"de ellos, en textos legales",elementos más largos que el límite
límite,,,,,,,
500,125,1717,389,543,1113,570,210
800,200,1253,560,775,649,278,43
1000,250,1080,651,967,476,184,20
1500,375,865,702,1429,261,87,11
2000,500,768,665,1895,164,46,6
3000,750,695,593,2712,91,22,1


Se usa un límite de **1,500 caracteres (unos 375 tokens) sin traslape**.

- **Cabe en los modelos de embeddings que leen hasta 512 tokens**, con margen; 2,000 caracteres en español pueden pasar ese límite.
- **Parte la mitad de artículos que 1,000 caracteres** (87 chunks legales que continúan un artículo, frente a 184), y solo 11 elementos son más largos que el límite, todos tablas.
- **La mediana casi no cambia por encima de 1,000**, porque la mayoría de los artículos son cortos: un límite mayor solo reduce cuántas veces se parte un artículo largo.

El límite cuenta los caracteres de los elementos; los saltos de línea que los separan dentro del chunk pueden sumar unos pocos más.

**Sin traslape**, porque los elementos no se cortan: no se pierde ninguna oración entre un chunk y el siguiente. El problema de los chunks que continúan un artículo es otro: no dicen de qué artículo son (por ejemplo, las fracciones XII a XX del artículo 3). Eso se resuelve con el contexto al inicio de cada chunk, que es la siguiente decisión.

In [4]:
MAX_CHARS = 1500
chunks = build_chunks(MAX_CHARS)
sizes = chunks["text"].str.len()
print(f"{len(chunks)} chunks; mediana de {int(sizes.median())} caracteres; {chunks['continues'].sum()} continúan un artículo o sección")
chunks.groupby("doc_id", sort=False).agg(
    chunks=("chunk_id", "size"),
    mediana=("text", lambda texts: int(texts.str.len().median())),
    continúan=("continues", "sum"),
    más_de_una_página=("pages", lambda pages: sum(len(p) > 1 for p in pages)),
)

865 chunks; mediana de 702 caracteres; 261 continúan un artículo o sección


,chunks,mediana,continúan,más_de_una_página
doc_id,,,,
lfpdppp,89,629,19,14
lgpdppso,185,637,42,28
reglamento_lfpdppp,161,512,15,27
inai_recomendaciones_ia,85,1147,48,13
unesco_ram_mexico,177,1268,124,34
principios_chapultepec,11,477,2,1
iniciativa_ley_ia_senado,157,397,11,24


## Decisión: contexto al inicio de cada chunk

Cada chunk empieza con una línea de contexto: el título del documento, el título de sección vigente y, si el chunk continúa un artículo, el número del artículo.

```
Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP) | Disposiciones Generales | Artículo 2 (continuación)
```

Por qué:

- **Los números de artículo no identifican la ley.** 137 de los 144 números de artículo aparecen en dos o más de los cuatro textos legales, y 62 en los cuatro.
- **Los chunks casi nunca nombran su documento.** Solo 84 de 865 lo hacen, así que una pregunta como «¿qué dice la LFPDPPP sobre…?» no encontraría los chunks de la propia ley por su nombre.
- **Los chunks que continúan un artículo no dicen cuál es.** De los 87 chunks legales que continúan un artículo o sección, 82 no contienen «Artículo N».

Las dos leyes y el reglamento llevan su sigla después del título, porque así suelen citarse. El artículo vigente lo fija cada «Artículo N» y lo reinicia cualquier otro título (por ejemplo, «Transitorios»); se agrega solo si el título de sección no lo dice ya. Así, 53 chunks reciben su artículo; de los otros 34 que continúan, 5 ya lo tienen en el título de sección y 29 son de los artículos transitorios («Tercero.-», «Quinto.-»), que no llevan número de artículo y quedan con «Transitorios» como título.

La línea de contexto va en el texto que se usa para buscar y en el que lee el LLM (`full_text`); el texto original queda en `text`. Se descartaron no agregar contexto, agregar solo título y sección (los chunks que continúan seguirían sin artículo) y pedir a un LLM que escriba el contexto de cada chunk (865 llamadas con costo y resultados no reproducibles; queda como mejora posible en producción).

In [5]:
SHORT_NAMES = {"lfpdppp": "LFPDPPP", "lgpdppso": "LGPDPPSO", "reglamento_lfpdppp": "Reglamento de la LFPDPPP"}


def context_line(chunk):
    title = MANIFEST[chunk["doc_id"]]["title"]
    if chunk["doc_id"] in SHORT_NAMES:
        title += f" ({SHORT_NAMES[chunk['doc_id']]})"
    parts = [title]
    heading = chunk["heading"] if isinstance(chunk["heading"], str) else ""
    article = chunk["article"] if isinstance(chunk["article"], str) else ""
    if heading:
        parts.append(heading)
    if chunk["continues"] and article and not heading.startswith(article):
        parts.append(f"{article} (continuación)")
    return " | ".join(parts)


chunks["context"] = chunks.apply(context_line, axis=1)
chunks["full_text"] = chunks["context"] + "\n\n" + chunks["text"]

with_article = chunks["context"].str.endswith("(continuación)")
print(f"Chunks con el artículo en el contexto: {with_article.sum()}")
print(f"Largo de la línea de contexto: mediana {int(chunks['context'].str.len().median())}, máximo {chunks['context'].str.len().max()} caracteres")
print(f"Largo de full_text: máximo {chunks['full_text'].str.len().max()} caracteres; "
      f"{(chunks['full_text'].str.len() > 2000).sum()} chunks pasan de 2,000")
for chunk_id in ["lfpdppp-0006", "lfpdppp-0046", "inai_recomendaciones_ia-0010"]:
    print(f"\n{chunk_id}: {chunks.set_index('chunk_id').at[chunk_id, 'context']}")

Chunks con el artículo en el contexto: 53
Largo de la línea de contexto: mediana 147, máximo 283 caracteres
Largo de full_text: máximo 4700 caracteres; 8 chunks pasan de 2,000

lfpdppp-0006: Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP) | Disposiciones Generales | Artículo 2 (continuación)

lfpdppp-0046: Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP) | Artículo 39. La Secretaría tiene las siguientes atribuciones:

inai_recomendaciones_ia-0010: Recomendaciones para el tratamiento de datos personales derivado del uso de la inteligencia artificial | INTRODUCCIÓN


## Decisión: notas al pie en el índice

Solo el INAI (90 notas) y UNESCO (198) tienen notas al pie. 84 son solo un enlace («Disponible en: URL [consulta: fecha]»): 77 del INAI y 7 de UNESCO. Las demás son sobre todo referencias (autor, año, título y enlace), pero unas 56 traen explicaciones o definiciones, como la de inteligencia artificial (nota 1 de UNESCO) o las de red neuronal, reconocimiento facial y cómputo en la nube (notas 28, 53 y 76 del INAI).

Se indexan las notas con texto, agrupadas por página:

- Una nota es «solo un enlace» si, al quitarle su número, los enlaces y las frases «Disponible en» y «[consulta: …]», no le quedan más de 5 caracteres. Esas no se indexan.
- Las notas de una misma página del PDF forman un chunk, hasta el mismo límite de 1,500 caracteres, sin cortar ninguna nota.
- Su línea de contexto es el título del documento y «Notas al pie, página N» (página del PDF).
- La columna `kind` distingue los chunks de texto (`texto`) de los de notas (`notas`).

Se descartaron no indexarlas (se perderían las definiciones), indexar también las que solo son un enlace (solo agregan URLs, y la búsqueda por palabras clave puede coincidir con palabras dentro de ellas) y hacer un chunk por nota (204 chunks muy cortos, que pueden quedar arriba solo por coincidir en una o dos palabras). La desventaja de agrupar por página es que las notas de una página tratan temas distintos.

In [6]:
LINK_PARTS = re.compile(r"https?://\S+|www\.\S+|Disponible en:?|\[?consulta:?[^\]]*\]?|Obtenido de", re.IGNORECASE)


def is_link_only(text):
    """True when almost nothing is left of a note after removing its number, its links and the citation phrases."""
    rest = LINK_PARTS.sub("", re.sub(r"^\d+\s+", "", text))
    return len(re.sub(r"[\s.,;:\[\]]+", " ", rest).strip()) <= 5


def note_record(doc_id, number, page, texts):
    return {
        "chunk_id": f"{doc_id}-nota-{number:03d}",
        "doc_id": doc_id,
        "pages": [int(page)],
        "heading": f"Notas al pie, página {int(page)}",
        "n_elements": len(texts),
        "continues": False,
        "article": None,
        "text": "\n\n".join(texts),
    }


def build_note_chunks(text_notes, max_chars):
    """Group the notes of each PDF page into chunks of up to max_chars characters, without cutting a note."""
    records = []
    for doc_id, doc_notes in text_notes.groupby("doc_id", sort=False):
        number = 0
        for page, page_notes in doc_notes.groupby("page"):
            group, size = [], 0
            for text in page_notes["text"]:
                if group and size + len(text) > max_chars:
                    number += 1
                    records.append(note_record(doc_id, number, page, group))
                    group, size = [], 0
                group.append(text)
                size += len(text)
            number += 1
            records.append(note_record(doc_id, number, page, group))
    return pd.DataFrame(records)


link_only = notes["text"].map(is_link_only)
note_chunks = build_note_chunks(notes[~link_only], MAX_CHARS)
note_chunks["context"] = note_chunks.apply(context_line, axis=1)
note_chunks["full_text"] = note_chunks["context"] + "\n\n" + note_chunks["text"]

text_chunks = chunks[~chunks["chunk_id"].str.contains("-nota-")]
chunks = pd.concat([text_chunks.assign(kind="texto"), note_chunks.assign(kind="notas")], ignore_index=True)

print(f"Notas: {len(notes)}; solo enlace: {link_only.sum()}; indexadas: {(~link_only).sum()}")
print(f"Chunks de notas: {len(note_chunks)}; mediana de {int(note_chunks['text'].str.len().median())} caracteres")
print(f"Chunks en total: {len(chunks)}")
print(f"\nEjemplo: {note_chunks.iloc[0]['context']}")
chunks.groupby(["doc_id", "kind"], sort=False).size().unstack(fill_value=0)

Notas: 288; solo enlace: 84; indexadas: 204
Chunks de notas: 49; mediana de 628 caracteres
Chunks en total: 914

Ejemplo: Recomendaciones para el tratamiento de datos personales derivado del uso de la inteligencia artificial | Notas al pie, página 11


kind,texto,notas
doc_id,,
lfpdppp,89,0
lgpdppso,185,0
reglamento_lfpdppp,161,0
inai_recomendaciones_ia,85,10
unesco_ram_mexico,177,39
principios_chapultepec,11,0
iniciativa_ley_ia_senado,157,0


## Decisión: tablas largas y chunks pequeños

**Tablas largas.** 11 elementos son más largos que el límite de 1,500 caracteres, y todos son tablas. La más larga es la Tabla 2 de UNESCO (p. 52, 4,564 caracteres). Muchos modelos de embeddings solo leen los primeros 512 tokens, así que el final de esas tablas no se vería al buscar.

Los chunks que tienen un elemento más largo que el límite se dividen:

- Primero entre filas de la tabla. Cada fila ya lleva el nombre de sus columnas, así que se entiende sola.
- Si una fila sigue siendo más larga que el límite (las tablas de una sola fila de las pp. 38 y 40 del INAI), se divide al final de una oración, sin separar un número de lista («4.») del texto que le sigue.
- Cada pieza conserva el contexto, las páginas y el título del chunk original. La primera conserva su identificador y las demás llevan `-2`, `-3`…

Se descartaron dejarlas enteras y dividir solo entre filas, porque las dos tablas de una sola fila seguirían pasando el límite.

**Chunks pequeños.** 44 chunks de texto tienen menos de 200 caracteres. 22 son artículos cortos completos, unos 20 son un título con un texto o elemento de lista corto, y uno es la contraportada de UNESCO («Síguenos @UNESCO #AI #HumanAI»). Al dividir las tablas aparecen 3 más, porque el final de una tabla puede quedar corto (por ejemplo, la última fila de la tabla de la p. 12 del INAI). Todos se quedan como están: la línea de contexto ya les agrega unos 150 caracteres, y unirlos al chunk siguiente mezclaría artículos, que es lo que la estrategia evita.

In [7]:
SENTENCE_BREAK = re.compile(r"(?<=[.;:!?])\s+")
LIST_MARK = re.compile(r"\(?(\d{1,3}|[IVX]{1,4}|[a-z])[.)]")


def sentences(row):
    """Split a row at sentence ends, keeping a list number such as "4." with the text that follows it."""
    merged = []
    for part in SENTENCE_BREAK.split(row):
        if merged and LIST_MARK.fullmatch(merged[-1]):
            merged[-1] += " " + part
        else:
            merged.append(part)
    return merged


def text_units(text, max_chars):
    """(separator, unit) pairs: the chunk's elements, and rows and sentences only inside elements longer than max_chars."""
    units = []
    for part in text.split("\n\n"):
        if len(part) <= max_chars:
            units.append(("\n\n", part))
            continue
        for row_number, row in enumerate(part.split("\n")):
            row_units = [row] if len(row) <= max_chars else sentences(row)
            for sentence_number, sentence in enumerate(row_units):
                if sentence_number > 0:
                    separator = " "
                elif row_number > 0:
                    separator = "\n"
                else:
                    separator = "\n\n"
                units.append((separator, sentence))
    return units


def split_text(text, max_chars):
    """Pieces of up to max_chars characters, cut only inside elements longer than the limit."""
    pieces, current = [], ""
    for separator, unit in text_units(text, max_chars):
        if current and len(current) + len(separator) + len(unit) > max_chars:
            pieces.append(current)
            current = unit
        else:
            current = current + separator + unit if current else unit
    if current:
        pieces.append(current)
    return pieces


def split_long_chunks(chunks, max_chars):
    records = []
    for chunk in chunks.to_dict("records"):
        has_long_element = any(len(part) > max_chars for part in chunk["text"].split("\n\n"))
        pieces = split_text(chunk["text"], max_chars) if has_long_element else [chunk["text"]]
        for number, piece in enumerate(pieces, start=1):
            record = dict(chunk, text=piece)
            if number > 1:
                record.update(chunk_id=f"{chunk['chunk_id']}-{number}", continues=True)
            record["context"] = context_line(record)
            record["full_text"] = record["context"] + "\n\n" + piece
            records.append(record)
    return pd.DataFrame(records)


before = len(chunks)
long_ids = chunks.loc[chunks["text"].map(lambda text: any(len(part) > MAX_CHARS for part in text.split("\n\n"))), "chunk_id"]
chunks = split_long_chunks(chunks, MAX_CHARS)
sizes = chunks["text"].str.len()
print(f"Chunks con una tabla más larga que el límite: {len(long_ids)}; chunks: {before} antes, {len(chunks)} después")
print(f"Texto más largo: {sizes.max()} caracteres; full_text más largo: {chunks['full_text'].str.len().max()}")
print(f"Chunks de texto con menos de 200 caracteres (se quedan): {((chunks['kind'] == 'texto') & (sizes < 200)).sum()}")
print("\nTabla 2 de UNESCO (p. 52):")
for _, chunk in chunks[chunks["chunk_id"].str.startswith(long_ids[long_ids.str.startswith("unesco")].iloc[-1])].iterrows():
    print(f"  {chunk['chunk_id']}: {len(chunk['text'])} caracteres, empieza con {chunk['text'][:70]!r}")

Chunks con una tabla más larga que el límite: 11; chunks: 914 antes, 927 después
Texto más largo: 1523 caracteres; full_text más largo: 1734
Chunks de texto con menos de 200 caracteres (se quedan): 47

Tabla 2 de UNESCO (p. 52):
  unesco_ram_mexico-0112: 1443 caracteres, empieza con 'RECOMENDACIONES PRINCIPALES\n\nRUBRO: 1. MAPA DEL ECOSISTEMA DE LA IA; R'
  unesco_ram_mexico-0112-2: 1138 caracteres, empieza con 'RUBRO: 4. EMITIR LA ESTRATEGIA NACIONAL DE IA; RECOMENDACIONES: 4. EMI'
  unesco_ram_mexico-0112-3: 775 caracteres, empieza con 'RUBRO: 4.2 ÉTICA, RESPONSABILIDAD E INCLUSIÓN. IA DE INTERÉS PÚBLICO; '
  unesco_ram_mexico-0112-4: 1234 caracteres, empieza con 'RUBRO: 4.3 ACTUALIZACIÓN EDUCATIVA Y RECONVERSIÓN LABORAL; RECOMENDACI'


## Consultas de evaluación

### Decisión: tipo de preguntas

Las 10 consultas son preguntas realistas: las que haría alguien que usa el sistema, por ejemplo un responsable de cumplimiento o un desarrollador que trabaja con datos personales. No se escogen para probar un tipo de contenido en particular. Al menos 2 son preguntas trampa, como pide el enunciado, redactadas también como las haría un usuario.

Se descartaron:

- **Preguntas dirigidas a los retos encontrados en el parsing y el chunking** (un artículo pedido por número y sigla, una definición dicha con otras palabras, una cifra de una tabla, una definición que solo está en una nota al pie…). Darían más material para el análisis de errores, pero no representan cómo se usaría el sistema.
- **Preguntas escritas a partir de pasajes elegidos al azar.** No tienen sesgo, pero suelen ser triviales y algunos documentos podrían quedarse sin preguntas.

La consecuencia es que puede que ninguna pregunta toque las tablas, las notas al pie o el texto escaneado de la iniciativa. El análisis de errores cubre lo que las preguntas sí tocan.

### Decisión: cómo se anotan las respuestas esperadas

Cada consulta se anota en `data/queries.json` con su pregunta, su respuesta ideal y los pasajes que la responden. Cada pasaje lleva el documento, la página del PDF y la cita exacta: un párrafo o una fila de tabla (o parte de ellos) copiados de `data/markdown/`.

```json
{
  "id": "q01",
  "question": "…",
  "type": "…",
  "ideal_answer": "…",
  "evidence": [{"doc_id": "lfpdppp", "page": 5, "quote": "…"}]
}
```

Los chunks esperados de una consulta son los de ese documento cuyo texto contiene alguna de sus citas (sin contar diferencias de espacios y saltos de línea). Una cita que no aparece en ningún chunk se avisa, para corregirla: puede ser una nota que solo es un enlace (no se indexan) o un pasaje que cruza de un chunk a otro.

Así, la anotación no depende de los identificadores de los chunks: si la segmentación cambia, las citas vuelven a encontrar los chunks correctos. Se descartaron anotar los identificadores de los chunks (habría que rehacer la anotación si cambian) y anotar solo documento y página (una página de una ley tiene 3 o 4 chunks, así que un chunk de la página correcta pero de otro artículo contaría como correcto).

Al probarlo con 105 pasajes copiados al azar de `data/markdown/`, todos aparecieron en algún chunk, salvo 4 notas al pie que solo son un enlace.

In [8]:
from IPython.display import display

QUERIES_FILE = DATA / "queries.json"


def normalize(text):
    return re.sub(r"\s+", " ", text).strip()


def expected_chunks(evidence, chunks):
    """Ids of the chunks of each quoted document that contain the quote, and the quotes found in no chunk."""
    normalized = chunks["text"].map(normalize)
    ids, missing = [], []
    for item in evidence:
        quote = normalize(item["quote"])
        found = chunks.loc[(chunks["doc_id"] == item["doc_id"]) & normalized.str.contains(quote, regex=False), "chunk_id"]
        if found.empty:
            missing.append(item)
        ids += [chunk_id for chunk_id in found if chunk_id not in ids]
    return ids, missing


if QUERIES_FILE.exists():
    queries = json.loads(QUERIES_FILE.read_text(encoding="utf-8"))
    for query in queries:
        query["expected_chunks"], missing = expected_chunks(query["evidence"], chunks)
        for item in missing:
            print(f"{query['id']}: cita no encontrada en {item['doc_id']}, p. {item['page']}: {item['quote'][:80]!r}")
    print(f"{len(queries)} consultas")
    display(pd.DataFrame(queries)[["id", "type", "question", "expected_chunks"]])
else:
    print("Todavía no hay consultas en data/queries.json")

Todavía no hay consultas en data/queries.json
